# Caught & Shared — Multi-Category Mentors
**Advanced User Control: Different Mentors for Different Life Categories**

In [ ]:
import torch
import torch.nn.functional as F
import pandas as pd
import numpy as np
import json
from datetime import datetime
from pathlib import Path

np.random.seed(42)
torch.manual_seed(42)

#### 1. Synthetic Data

In [ ]:
num_users = 20
num_items = 100

CATEGORIES = ["sneakers", "outerwear", "casual", "tech", "home", "workwear"] # or whatever

user_embs = torch.randn(num_users, 64)
item_embs = torch.randn(num_items, 64)

interactions = []
for u in range(num_users):
    num_inter = np.random.randint(5, 15)
    items = np.random.choice(num_items, num_inter, replace=False)
    for i in items:
        interactions.append((u, i))

inter_df = pd.DataFrame(interactions, columns=['user_id', 'item_id'])

print(f"{len(inter_df)} interactions | {len(CATEGORIES)} categories")

#### 2. Multi-Category Mentor Storage

In [ ]:
# Global storage: {user_id: {category: {"mentor_id": int, "preset": str, "alpha": float, "added_at": str}}}
user_category_mentors = {}
PREFERENCES_FILE = Path("user_category_mentors.json")


def load_category_preferences():
    global user_category_mentors
    if PREFERENCES_FILE.exists():
        with open(PREFERENCES_FILE, 'r', encoding='utf-8') as f:
            user_category_mentors = json.load(f)
        print(f"Loaded category preferences for {len(user_category_mentors)} users")
    else:
        print("No saved category preferences found.")


def save_category_preferences():
    with open(PREFERENCES_FILE, 'w', encoding='utf-8') as f:
        json.dump(user_category_mentors, f, indent=2, ensure_ascii=False)

def assign_mentor_to_category(user_id: int, category: str, mentor_id: int, preset: str = "Balanced"):
    """Assign a mentor to a specific category"""
    if user_id not in user_category_mentors:
        user_category_mentors[user_id] = {}
    
    alpha = INFLUENCE_PRESETS[preset]["alpha"]
    
    user_category_mentors[user_id][category] = {
        "mentor_id": mentor_id,
        "preset": preset,
        "alpha": round(alpha, 2),
        "added_at": datetime.now().strftime("%Y-%m-%d %H:%M")
    }
    save_category_preferences()
    print(f"Assigned Mentor {mentor_id} ({preset}) to category '{category}' for User {user_id}")


def remove_mentor_from_category(user_id: int, category: str):
    if user_id in user_category_mentors and category in user_category_mentors[user_id]:
        del user_category_mentors[user_id][category]
        if not user_category_mentors[user_id]:
            del user_category_mentors[user_id]
        save_category_preferences()
        print(f"Removed mentor from category '{category}' for User {user_id}")


def get_user_category_config(user_id: int):
    """Return all category mentors for a user"""
    return user_category_mentors.get(user_id, {})


load_category_preferences()

#### 3. Category-Aware Recommendation Function

In [ ]:
@torch.no_grad()
def get_recommendations_category_aware(user_id: int, current_category: str = None, k=8):
    """
    If current_category is specified → use mentor for that category
    If None → blend all active category mentors
    """
    user_emb = user_embs[user_id].clone()
    category_config = get_user_category_config(user_id)
    
    if not category_config:
        final_emb = user_emb
    else:
        final_emb = user_emb.clone()
        
        if current_category and current_category in category_config:
            info = category_config[current_category]
            mentor_id = info["mentor_id"]
            alpha = info["alpha"]
            
            mentor_emb = user_embs[mentor_id]
            final_emb += alpha * (mentor_emb - user_emb)
            
        else:
            for cat, info in category_config.items():
                mentor_id = info["mentor_id"]
                alpha = info["alpha"]
                mentor_emb = user_embs[mentor_id]
                final_emb += alpha * (mentor_emb - user_emb) / len(category_config)
        
        final_emb = F.normalize(final_emb, p=2, dim=0)
    
    scores = torch.matmul(final_emb.unsqueeze(0), item_embs.T).squeeze(0)
    topk_scores, topk_indices = torch.topk(scores, k=k)
    
    return {
        'user_id': user_id,
        'current_category': current_category,
        'recommended_items': topk_indices.tolist(),
        'scores': topk_scores.tolist()
    }

In [ ]:

assign_mentor_to_category(0, "sneakers", 5, preset="Deep")
assign_mentor_to_category(0, "outerwear", 7, preset="Balanced")
assign_mentor_to_category(0, "tech", 12, preset="Light")
assign_mentor_to_category(0, "home", 9, preset="Balanced")

config = get_user_category_config(0)
print(f"\nUser 0's Personal Mentor Team:")
for cat, info in config.items():
    print(f"   {cat:12} → Mentor {info['mentor_id']:2d} ({info['preset']})")

print("\nRecommendations:")
for cat in ["sneakers", "outerwear", "tech"]:
    rec = get_recommendations_category_aware(user_id=0, current_category=cat, k=5)
    print(f"   {cat:12} → {rec['recommended_items']}")